# Versão 2 — comparação de janelas e análise hidrológica

Prevemos a cota média diária em `t+1` usando janelas de cotas e chuva observadas. A V1 permanece em `analise_redes_neurais.ipynb`; esta versão salva suas saídas em `resultados/v2/`. O corte temporal da V1 será tratado como exploratório, pois já foi consultado.

Os dois protocolos de chuva distinguem dados até `t` e até `t−1`. Janelas com lacunas ou medições inválidas são descartadas. As comparações de cada protocolo usam exatamente as mesmas datas entre J7, J15 e J30.

## 1. Ambiente, caminhos e reprodutibilidade

Execute de cima para baixo em um kernel limpo. A arquitetura é selecionada pela MAE de validação; o teste não é usado para seleção. Os scalers são ajustados somente no treino.

In [1]:
from pathlib import Path
import sys, warnings, json, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEMENTE = 42
np.random.seed(SEMENTE)
tf.keras.utils.set_random_seed(SEMENTE)
try:
    tf.config.threading.set_intra_op_parallelism_threads(2)
    tf.config.threading.set_inter_op_parallelism_threads(2)
except RuntimeError:
    pass

def localizar_projeto():
    candidatos = [Path.cwd(), Path.cwd().parent]
    try: candidatos.append(Path(__file__).resolve().parents[1])
    except NameError: pass
    for pasta in candidatos:
        if (pasta / 'dados').is_dir() and (pasta / 'notebooks').is_dir(): return pasta.resolve()
    raise FileNotFoundError('Abra o projeto ou execute a partir da pasta RedesNeurais_ANA.')

RAIZ = localizar_projeto()
DIR_DADOS = RAIZ / 'dados'
DIR_RESULTADOS = RAIZ / 'resultados' / 'v2'
DIR_FIGURAS = DIR_RESULTADOS / 'figuras'
DIR_RESULTADOS.mkdir(parents=True, exist_ok=True)
DIR_FIGURAS.mkdir(parents=True, exist_ok=True)
VERSOES = {'python': platform.python_version(), 'pandas': pd.__version__, 'numpy': np.__version__, 'scikit-learn': sklearn.__version__, 'tensorflow': tf.__version__, 'seed': SEMENTE}
(DIR_RESULTADOS / 'versoes.json').write_text(json.dumps(VERSOES, indent=2), encoding='utf-8')
print(VERSOES)
print('Projeto:', RAIZ)
print('Saídas V2:', DIR_RESULTADOS)

{'python': '3.13.5', 'pandas': '2.3.3', 'numpy': '2.1.3', 'scikit-learn': '1.9.1', 'tensorflow': '2.20.0', 'seed': 42}
Projeto: C:\Users\PedroMenezes\Documents\EngenhariaDeComputacao\DeepLearning\RedesNeurais_ANA
Saídas V2: C:\Users\PedroMenezes\Documents\EngenhariaDeComputacao\DeepLearning\RedesNeurais_ANA\resultados\v2


## 2. Leitura e auditoria das exportações ANA

As linhas de metadados anteriores ao cabeçalho são mantidas para auditoria. A expansão diária respeita status `1`, `MediaDiaria=1` para cota e vazão, calendário real e prioridade de consistência `2`. Ausências nunca são convertidas em zero.

In [2]:
ARQUIVOS = {'Cota': DIR_DADOS / '86720000_Cotas.csv', 'Vazao': DIR_DADOS / '86720000_Vazoes.csv', 'Chuva': DIR_DADOS / '2951010_Chuvas.csv'}
def ler_exportacao_ana(caminho):
    if not caminho.is_file(): raise FileNotFoundError(f'Arquivo obrigatório ausente: {caminho}')
    conteudo = caminho.read_bytes(); texto = None; codificacao = None
    for candidata in ('utf-8-sig', 'cp1252', 'latin1'):
        try: texto = conteudo.decode(candidata); codificacao = candidata; break
        except UnicodeDecodeError: continue
    linhas = texto.splitlines()
    indice = next((i for i, linha in enumerate(linhas) if linha.lstrip('\ufeff').startswith('EstacaoCodigo;')), None)
    if indice is None: raise ValueError(f'Cabeçalho ANA não encontrado em {caminho.name}')
    tabela = pd.read_csv(caminho, sep=';', encoding=codificacao, skiprows=indice, dtype='string', skipinitialspace=True)
    tabela.columns = tabela.columns.str.strip()
    return tabela, codificacao, indice + 1, linhas[:indice]

def para_numero(serie):
    return pd.to_numeric(serie.astype('string').str.strip().str.replace(',', '.', regex=False), errors='coerce')

brutos, auditoria_arquivos, metadados = {}, [], {}
for nome, caminho in ARQUIVOS.items():
    tabela, codificacao, linha_cabecalho, linhas_meta = ler_exportacao_ana(caminho)
    brutos[nome] = tabela; metadados[nome] = linhas_meta
    datas = pd.to_datetime(tabela['Data'], dayfirst=True, errors='coerce')
    col_dia = {'Cota':'Cota01', 'Vazao':'Vazao01', 'Chuva':'Chuva01'}[nome]
    auditoria_arquivos.append({'Série':nome,'Arquivo':caminho.name,'Codificação':codificacao,'Linha cabeçalho':linha_cabecalho,'Linhas mensais':len(tabela),'Colunas':len(tabela.columns),'Primeira data':datas.min(),'Última data':datas.max(),'Coluna diária presente':col_dia in tabela})
display(pd.DataFrame(auditoria_arquivos))
print('Metadados de cotas:', *metadados['Cota'], sep='\n')

Metadados de cotas:
Agência Nacional de Águas - ANA
Superintendência de Gestão da Rede Hidrometeorológica-SGH
Sistema de Informações Hidrológicas - Versão WEB
Fone: (61) 2109-5242
E-mail: hidro@ana.gov.br

NivelConsistencia: 1 = Bruto, 2 = Consistido
MediaDiaria: Indica se a medição é média diaria ou instantânea.  0 = Não , 1 = Sim
TipoMedicaoCotas: 1 = Escala, 2 = Linígrafo, 3 = Data Logger, 5 = Escala SMS
Status: 0 = Branco, 1 = Real, 2 = Estimado, 3 = Duvidoso, 4 = Régua Seca

Restrições da consulta:
Código da Estação:86720000




,Série,Arquivo,Codificação,Linha cabeçalho,Linhas mensais,Colunas,Primeira data,Última data,Coluna diária presente
0,Cota,86720000_Cotas.csv,cp1252,16,2087,78,1941-10-01,2026-04-01,True
1,Vazao,86720000_Vazoes.csv,cp1252,16,1012,78,1941-10-01,2026-04-01,True
2,Chuva,2951010_Chuvas.csv,cp1252,15,1701,75,1943-04-01,2026-04-01,True


In [3]:
def expandir_diario(nome, tabela):
    prefixo = nome; mensal = tabela.copy()
    mensal['Data'] = pd.to_datetime(mensal['Data'], dayfirst=True, errors='coerce')
    mensal['NivelConsistencia'] = para_numero(mensal['NivelConsistencia'])
    if nome in ('Cota','Vazao'):
        mensal['MediaDiaria'] = para_numero(mensal['MediaDiaria'])
        mensal = mensal[mensal['MediaDiaria'].eq(1)]
    blocos = []
    for dia in range(1,32):
        coluna = f'{prefixo}{dia:02d}'; status_col = f'{coluna}Status'
        if coluna not in mensal: continue
        data_dia = mensal['Data'] + pd.to_timedelta(dia-1, unit='D')
        bloco = pd.DataFrame({'Data':data_dia,'Valor':para_numero(mensal[coluna]),'Status':mensal[status_col].astype('string').str.strip() if status_col in mensal else pd.NA,'Consistencia':mensal['NivelConsistencia']})
        blocos.append(bloco.loc[data_dia.dt.month.eq(mensal['Data'].dt.month)])
    longo = pd.concat(blocos, ignore_index=True).dropna(subset=['Data','Valor'])
    elegivel = longo[longo['Status'].eq('1') & np.isfinite(longo['Valor'])].copy()
    if elegivel.empty: raise ValueError(f'Sem valores elegíveis para {nome}')
    colisoes = int(elegivel.groupby('Data').size().gt(1).sum())
    elegivel = elegivel.sort_values(['Data','Consistencia'], ascending=[True,False])
    melhor = elegivel.groupby('Data')['Consistencia'].transform('max')
    elegivel = elegivel[elegivel['Consistencia'].eq(melhor)]
    diaria = elegivel.groupby('Data')['Valor'].median().sort_index().rename(nome)
    resumo = {'Série':nome,'Observações com valor':len(longo),'Elegíveis após status':len(elegivel),'Dias únicos':len(diaria),'Dias com duplicatas elegíveis':colisoes,'Início':diaria.index.min(),'Fim':diaria.index.max(),'Lacunas':int((diaria.index.to_series().diff().dt.days > 1).sum())}
    return diaria, resumo, longo

series, resumos, dados_longos = {}, [], {}
for nome, tabela in brutos.items():
    series[nome], resumo, dados_longos[nome] = expandir_diario(nome,tabela); resumos.append(resumo)
display(pd.DataFrame(resumos))

base = pd.concat(series.values(), axis=1).sort_index(); base.index.name = 'Data'
base.to_csv(DIR_RESULTADOS / 'series_diarias.csv', date_format='%Y-%m-%d')
calendario = pd.date_range(base.index.min(), base.index.max(), freq='D', name='Data')
diaria = base.reindex(calendario)
print('Cobertura série diária:'); display(base.notna().sum().rename('dias observados').to_frame())

# Auditoria específica do evento de maio de 2024: valor e status nas linhas originais.
def auditoria_evento(tabela, coluna, data='01/05/2024'):
    campos = [c for c in ['Data','NivelConsistencia','hora','Hora','MediaDiaria','TipoMedicaoCotas',coluna,f'{coluna}Status'] if c in tabela.columns]
    return tabela.loc[tabela['Data'].eq(data), campos]
evento_cota = auditoria_evento(brutos['Cota'],'Cota02')
evento_chuva = auditoria_evento(brutos['Chuva'],'Chuva02')
display(evento_cota); display(evento_chuva)
evento_cota.to_csv(DIR_RESULTADOS / 'auditoria_evento_maio_2024_cota.csv', index=False)
evento_chuva.to_csv(DIR_RESULTADOS / 'auditoria_evento_maio_2024_chuva.csv', index=False)

Cobertura série diária:


,Série,Observações com valor,Elegíveis após status,Dias únicos,Dias com duplicatas elegíveis,Início,Fim,Lacunas
0,Cota,46153,21193,21193,15595,1941-10-15,2026-04-30,236
1,Vazao,24619,20103,20103,0,1941-10-15,2026-04-30,250
2,Chuva,51261,28844,28844,22338,1943-04-19,2026-04-30,71


,dias observados
Cota,21193
Vazao,20103
Chuva,28844


,Data,NivelConsistencia,hora,MediaDiaria,TipoMedicaoCotas,Cota02,Cota02Status
69,01/05/2024,1,07:00,0,3,<NA>,0
70,01/05/2024,1,17:00,0,3,2314,1
71,01/05/2024,1,<NA>,1,3,<NA>,0


,Data,NivelConsistencia,Chuva02,Chuva02Status


## 3. Constru??o de janelas v?lidas e divis?o temporal

Para cada janela, as cotas usam `t?(n?1)?t`. No cen?rio de chuva at? `t`, a chuva usa os mesmos dias; no cen?rio conservador, usa `t?n?t?1`. O alvo ? sempre a cota de `t+1`. J7/J15/J30 compartilham a interse??o de datas eleg?veis. Para preservar a avalia??o explorat?ria da V1, usamos seus limites exatos: treino at? 09/03/1993, valida??o de 10/03/1993 a 24/11/2008 e teste a partir de 25/11/2008.

In [4]:
JANELAS = (7,15,30)
cota_array = diaria['Cota'].to_numpy(dtype=float, na_value=np.nan)
chuva_array = diaria['Chuva'].to_numpy(dtype=float, na_value=np.nan)
finite_cota = np.isfinite(cota_array)
finite_chuva = np.isfinite(chuva_array)

def mascara_janela(valida, n):
    cumulativa = np.r_[0, np.cumsum(valida, dtype=np.int64)]
    fim = np.arange(len(valida))
    inicio = fim - n + 1
    ok = inicio >= 0
    ok[ok] &= (cumulativa[fim[ok]+1] - cumulativa[inicio[ok]]) == n
    return ok

def construir_amostras(n, fim_chuva):
    deslocamento = 0 if fim_chuva == 't' else 1
    # i ? t; para o protocolo t-1, a chuva termina em i-1.
    chuva_ok = mascara_janela(finite_chuva, n)
    if deslocamento:
        chuva_ok = np.r_[False, chuva_ok[:-1]]
    cota_ok = mascara_janela(finite_cota, n)
    alvo_ok = np.r_[finite_cota[1:], False]
    valido = cota_ok & chuva_ok & alvo_ok
    valido[:n-1+deslocamento] = False
    indices = np.flatnonzero(valido)
    linhas = []
    for i in indices:
        inicio_cota = i-n+1
        fim_chuva_idx = i-deslocamento
        inicio_chuva = fim_chuva_idx-n+1
        linhas.append({'t':calendario[i], 'data_alvo':calendario[i+1],
                       'cotas':cota_array[inicio_cota:i+1].copy(),
                       'chuvas':chuva_array[inicio_chuva:fim_chuva_idx+1].copy(),
                       'alvo':float(cota_array[i+1]), 'persistencia':float(cota_array[i])})
    return linhas

amostras = {(n,c):construir_amostras(n,c) for n in JANELAS for c in ('t','t-1')}
contagens_elegiveis = {(n,c):len(amostras[(n,c)]) for n in JANELAS for c in ('t','t-1')}
for c in ('t','t-1'):
    sets = [set(pd.Timestamp(r['t']) for r in amostras[(n,c)]) for n in JANELAS]
    comuns = set.intersection(*sets)
    for n in JANELAS: amostras[(n,c)] = [r for r in amostras[(n,c)] if pd.Timestamp(r['t']) in comuns]
    print(f'Chuva at? {c}: {len(comuns):,} datas comuns; eleg?veis antes da interse??o: {[contagens_elegiveis[(n,c)] for n in JANELAS]}')

def empacotar(linhas, n, uso_chuva):
    features = [f'cota_d{i-n+1}' for i in range(n)]
    if uso_chuva: features += [f'chuva_d{i-n+1}' for i in range(n)]
    X = np.vstack([np.r_[r['cotas'], r['chuvas'] if uso_chuva else []] for r in linhas])
    return X, np.array([r['alvo'] for r in linhas]), np.array([r['persistencia'] for r in linhas]), pd.DatetimeIndex([r['t'] for r in linhas]), pd.DatetimeIndex([r['data_alvo'] for r in linhas]), features

def dividir(X,y,persist,t,alvos,features):
    # Limites id?nticos aos alvos de treino/valida??o/teste informados na V1.
    fatias={
        'treino': alvos <= pd.Timestamp('1993-03-09'),
        'validacao': (alvos >= pd.Timestamp('1993-03-10')) & (alvos <= pd.Timestamp('2008-11-24')),
        'teste': alvos >= pd.Timestamp('2008-11-25'),
    }
    if any(mask.sum()==0 for mask in fatias.values()): raise ValueError('Um split ficou sem amostras; confira as datas comuns.')
    return {nome:{'X':X[mask],'y':y[mask],'persist':persist[mask],'t':t[mask],'alvos':alvos[mask],'features':features} for nome,mask in fatias.items()}

particoes = {} ; cobertura = []
for chuva_fim in ('t','t-1'):
  for n in JANELAS:
    linhas=amostras[(n,chuva_fim)]
    for com_chuva in (False,True):
      X,y,p,t,alvos,features=empacotar(linhas,n,com_chuva)
      particoes[(chuva_fim,n,com_chuva)] = dividir(X,y,p,t,alvos,features)
    p=particoes[(chuva_fim,n,False)]
    cobertura.append({'Chuva termina em':chuva_fim,'Janela':n,'N comum':len(linhas),'In?cio t':p['treino']['t'][0],'Fim alvo':p['teste']['alvos'][-1],'N treino':len(p['treino']['y']),'N valida??o':len(p['validacao']['y']),'N teste':len(p['teste']['y'])})
cobertura = pd.DataFrame(cobertura); display(cobertura)
cobertura.to_csv(DIR_RESULTADOS / 'cobertura_e_splits.csv', index=False, date_format='%Y-%m-%d')

cobertura_anual=[]
for c in ('t','t-1'):
  for n in JANELAS:
    anos=pd.Series([r['data_alvo'].year for r in amostras[(n,c)]]).value_counts().sort_index()
    cobertura_anual.extend({'Chuva termina em':c,'Janela':n,'Ano':ano,'N':qtd} for ano,qtd in anos.items())
pd.DataFrame(cobertura_anual).to_csv(DIR_RESULTADOS / 'cobertura_anual.csv',index=False)


Chuva at? t: 15,673 datas comuns; eleg?veis antes da interse??o: [18252, 17162, 15673]
Chuva at? t-1: 15,667 datas comuns; eleg?veis antes da interse??o: [18245, 17155, 15667]


,Chuva termina em,Janela,N comum,In?cio t,Fim alvo,N treino,N valida??o,N teste
0,t,7,15673,1943-05-18,2026-04-30,11660,1762,2251
1,t,15,15673,1943-05-18,2026-04-30,11660,1762,2251
2,t,30,15673,1943-05-18,2026-04-30,11660,1762,2251
3,t-1,7,15667,1943-05-19,2026-04-30,11659,1761,2247
4,t-1,15,15667,1943-05-19,2026-04-30,11659,1761,2247
5,t-1,30,15667,1943-05-19,2026-04-30,11659,1761,2247


## 4. Seleção de modelos e avaliação exploratória

Em cada janela e cenário, são testadas MLPs 32 e 64–32 para cotas e para cotas+chuva. Early stopping monitora validação. Persistência e Ridge são avaliados nas mesmas datas. O modelo final é escolhido pela MAE de validação separadamente para cada combinação de janela e conjunto de entradas.

In [5]:
def criar_mlp(n_features, arquitetura):
    camadas=[keras.layers.Input(shape=(n_features,))]
    camadas += [keras.layers.Dense(u,activation='relu') for u in arquitetura]
    camadas += [keras.layers.Dense(1,activation='linear')]
    modelo=keras.Sequential(camadas)
    modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),loss=keras.losses.Huber())
    return modelo
def metricas(real,pred):
    return {'MAE':mean_absolute_error(real,pred),'RMSE':np.sqrt(mean_squared_error(real,pred)),'R2':r2_score(real,pred)}

resultados_modelos=[]; detalhes_modelos=[]; previsoes={}; historicos=[]
for chuva_fim in ('t','t-1'):
  for n in JANELAS:
    for com_chuva in (False,True):
      p=particoes[(chuva_fim,n,com_chuva)]; tr,va,te=p['treino'],p['validacao'],p['teste']
      sx=MinMaxScaler(); sy=MinMaxScaler()
      xtr=sx.fit_transform(tr['X']); xva=sx.transform(va['X']); xte=sx.transform(te['X'])
      ytr=sy.fit_transform(tr['y'].reshape(-1,1)); yva=sy.transform(va['y'].reshape(-1,1))
      family='cota+chuva' if com_chuva else 'cota'
      candidatos=[]
      for arquitetura in ((32,),(64,32)):
        tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(SEMENTE)
        modelo=criar_mlp(xtr.shape[1],arquitetura)
        parada=keras.callbacks.EarlyStopping(monitor='val_loss',patience=12,min_delta=1e-5,restore_best_weights=True)
        hist=modelo.fit(xtr,ytr,validation_data=(xva,yva),epochs=120,batch_size=128,shuffle=False,callbacks=[parada],verbose=0)
        predva=sy.inverse_transform(modelo.predict(xva,verbose=0)).ravel()
        maeva=mean_absolute_error(va['y'],predva)
        nome=f'MLP {arquitetura} {family}'
        candidatos.append({'modelo':modelo,'sx':sx,'sy':sy,'nome':nome,'arquitetura':arquitetura,'mae_val':maeva,'hist':hist.history,'epocas':len(hist.history['loss'])})
      escolhido=min(candidatos,key=lambda z:z['mae_val'])
      for cand in candidatos:
        detalhes_modelos.append({'Chuva termina em':chuva_fim,'Janela':n,'Entradas':family,'Arquitetura':str(cand['arquitetura']),'MAE validação':cand['mae_val'],'Épocas':cand['epocas'],'Selecionado':cand is escolhido,'N treino':len(tr['y']),'N validação':len(va['y']),'N teste':len(te['y']),'Início treino':str(tr['alvos'][0].date()),'Fim treino':str(tr['alvos'][-1].date()),'Início validação':str(va['alvos'][0].date()),'Fim validação':str(va['alvos'][-1].date()),'Início teste':str(te['alvos'][0].date()),'Fim teste':str(te['alvos'][-1].date())})
        for ep,(lt,lv) in enumerate(zip(cand['hist']['loss'],cand['hist']['val_loss']),1): historicos.append({'Chuva termina em':chuva_fim,'Janela':n,'Entradas':family,'Arquitetura':str(cand['arquitetura']),'Época':ep,'Perda treino':lt,'Perda validação':lv})
      pred=escolhido['sy'].inverse_transform(escolhido['modelo'].predict(xte,verbose=0)).ravel()
      chave=(chuva_fim,n,com_chuva); previsoes[chave]={'real':te['y'],'pred':pred,'datas':te['alvos'],'treino':tr,'teste':te,'modelo':escolhido}
      r=metricas(te['y'],pred); resultados_modelos.append({'Protocolo':f'Chuva até {chuva_fim}','Janela':n,'Modelo':escolhido['nome'],'Entradas':family,'N treino':len(tr['y']),'N validação':len(va['y']),'N teste':len(te['y']),**r})
      pred_p=te['persist']; rp=metricas(te['y'],pred_p); resultados_modelos.append({'Protocolo':f'Chuva até {chuva_fim}','Janela':n,'Modelo':'Persistência','Entradas':'cota(t)','N treino':len(tr['y']),'N validação':len(va['y']),'N teste':len(te['y']),**rp})
      ridge=Ridge(alpha=1.0); ridge.fit(sx.transform(tr['X']),tr['y']); pred_r=ridge.predict(xte); rr=metricas(te['y'],pred_r)
      resultados_modelos.append({'Protocolo':f'Chuva até {chuva_fim}','Janela':n,'Modelo':f'Ridge {family}','Entradas':family,'N treino':len(tr['y']),'N validação':len(va['y']),'N teste':len(te['y']),**rr})
      previsoes[chave]['ridge']=pred_r; previsoes[chave]['persist']=pred_p

df_resultados=pd.DataFrame(resultados_modelos); df_detalhes=pd.DataFrame(detalhes_modelos); df_hist=pd.DataFrame(historicos)
display(df_resultados); display(df_detalhes)
df_resultados.to_csv(DIR_RESULTADOS/'metricas_modelos.csv',index=False)
df_detalhes.to_csv(DIR_RESULTADOS/'selecao_modelos_validacao.csv',index=False)
df_hist.to_csv(DIR_RESULTADOS/'historico_treinamento.csv',index=False)
for c in ('t','t-1'):
  tabs=[]
  for n in JANELAS:
    v=previsoes[(c,n,True)]
    parte=pd.DataFrame({'janela':n,'data_alvo':v['datas'],'cota_real':v['real'],'mlp_cota_chuva':v['pred'],'ridge':v['ridge'],'persistencia':v['persist']})
    tabs.append(parte)
  pd.concat(tabs,ignore_index=True).to_csv(DIR_RESULTADOS/f'predicoes_teste_chuva_{c}.csv',index=False,date_format='%Y-%m-%d')

,Protocolo,Janela,Modelo,Entradas,N treino,N validação,N teste,MAE,RMSE,R2
0,Chuva até t,7,"MLP (64, 32) cota",cota,11660,1762,2251,29.239349,70.406661,0.726150
1,Chuva até t,7,Persistência,cota(t),11660,1762,2251,31.012883,78.737427,0.657510
2,Chuva até t,7,Ridge cota,cota,11660,1762,2251,27.760157,74.014612,0.697364
3,Chuva até t,7,"MLP (32,) cota+chuva",cota+chuva,11660,1762,2251,31.515061,70.139509,0.728224
4,Chuva até t,7,Persistência,cota(t),11660,1762,2251,31.012883,78.737427,0.657510
5,Chuva até t,7,Ridge cota+chuva,cota+chuva,11660,1762,2251,29.693622,71.963650,0.713904
6,Chuva até t,15,"MLP (64, 32) cota",cota,11660,1762,2251,29.317020,74.532382,0.693115
7,Chuva até t,15,Persistência,cota(t),11660,1762,2251,31.012883,78.737427,0.657510
8,Chuva até t,15,Ridge cota,cota,11660,1762,2251,28.082974,73.831721,0.698858
9,Chuva até t,15,"MLP (32,) cota+chuva",cota+chuva,11660,1762,2251,32.358184,71.405661,0.718324


,Chuva termina em,Janela,Entradas,Arquitetura,MAE validação,Épocas,Selecionado,N treino,N validação,N teste,Início treino,Fim treino,Início validação,Fim validação,Início teste,Fim teste
0,t,7,cota,"(32,)",37.412580,30,False,11660,1762,2251,1943-05-19,1993-03-09,1993-03-10,2008-09-22,2008-11-28,2026-04-30
1,t,7,cota,"(64, 32)",35.162206,24,True,11660,1762,2251,1943-05-19,1993-03-09,1993-03-10,2008-09-22,2008-11-28,2026-04-30
2,t,7,cota+chuva,"(32,)",36.130581,47,True,11660,1762,2251,1943-05-19,1993-03-09,1993-03-10,2008-09-22,2008-11-28,2026-04-30
3,t,7,cota+chuva,"(64, 32)",36.583575,19,False,11660,1762,2251,1943-05-19,1993-03-09,1993-03-10,2008-09-22,2008-11-28,2026-04-30
4,t,15,cota,"(32,)",37.433928,39,False,11660,1762,2251,1943-05-19,1993-03-09,1993-03-10,2008-09-22,2008-11-28,2026-04-30
5,t,15,cota,"(64, 32)",35.191806,14,True,11660,1762,2251,1943-05-19,1993-03-09,1993-03-10,2008-09-22,2008-11-28,2026-04-30
6,t,15,cota+chuva,"(32,)",39.374252,32,True,11660,1762,2251,1943-05-19,1993-03-09,1993-03-10,2008-09-22,2008-11-28,2026-04-30
7,t,15,cota+chuva,"(64, 32)",40.319236,20,False,11660,1762,2251,1943-05-19,1993-03-09,1993-03-10,2008-09-22,2008-11-28,2026-04-30
8,t,30,cota,"(32,)",36.216077,34,True,11660,1762,2251,1943-05-19,1993-03-09,1993-03-10,2008-09-22,2008-11-28,2026-04-30
9,t,30,cota,"(64, 32)",37.369204,21,False,11660,1762,2251,1943-05-19,1993-03-09,1993-03-10,2008-09-22,2008-11-28,2026-04-30


## 5. Figuras e análise por nível

Os gráficos principais usam o protocolo de chuva até `t`, com as mesmas datas de teste entre janelas. A unidade é mantida como aparece nas exportações; a conversão para metros só é usada no comentário de comparação com o levantamento do SGB e não é aplicada às métricas sem validação formal da unidade. P10/P90 são calculados somente no treino.

In [6]:
# As três janelas no protocolo escolhido por convenção de disponibilidade mais ampla: chuva até t.
pred_todas={n:previsoes[('t',n,True)] for n in JANELAS}
datas_comuns=set.intersection(*[set(v['datas']) for v in pred_todas.values()])
datas_comuns=pd.DatetimeIndex(sorted(datas_comuns))
for n in JANELAS:
    v=pred_todas[n]; idx=pd.Index(v['datas']).get_indexer(datas_comuns);
    fig,ax=plt.subplots(figsize=(13,5)); ax.plot(datas_comuns,v['real'][idx],label='Cota observada',lw=1.1); ax.plot(datas_comuns,v['pred'][idx],label='MLP prevista',lw=1.0)
    ax.set(title=f'Cota observada e previsão — janela {n} dias (teste exploratório)',xlabel='Data do alvo',ylabel='Cota (unidade da exportação ANA)'); ax.grid(alpha=.25); ax.legend(); fig.tight_layout(); fig.savefig(DIR_FIGURAS/f'comparacao_{n}_dias.png',dpi=160,bbox_inches='tight'); plt.show()

fig,axes=plt.subplots(1,3,figsize=(15,4))
metricas_fig=[('MAE','MAE (menor é melhor)'),('RMSE','RMSE (menor é melhor)'),('R2','R² (maior é melhor)')]
for ax,(col,titulo) in zip(axes,metricas_fig):
    q=df_resultados[(df_resultados['Protocolo']=='Chuva até t') & (df_resultados['Entradas']=='cota+chuva') & (df_resultados['Modelo'].str.startswith('MLP'))].sort_values('Janela')
    ax.plot(q['Janela'],q[col],marker='o'); ax.set(title=titulo,xlabel='Janela (dias)',xticks=JANELAS); ax.grid(alpha=.25)
fig.suptitle('Comparação das janelas — MLP com cotas e chuva'); fig.tight_layout(); fig.savefig(DIR_FIGURAS/'comparacao_janelas_metricas.png',dpi=160,bbox_inches='tight'); plt.show()

# Sobreposição das previsões selecionadas no mesmo episódio/datas.
fig,ax=plt.subplots(figsize=(13,5));
for n,v in pred_todas.items():
    idx=pd.Index(v['datas']).get_indexer(datas_comuns); ax.plot(datas_comuns,v['pred'][idx],label=f'J{n}',lw=.9)
v=pred_todas[7]; idx=pd.Index(v['datas']).get_indexer(datas_comuns); ax.plot(datas_comuns,v['real'][idx],color='black',label='Observado',lw=1.2)
ax.set(title='Sobreposição das previsões nas mesmas datas de teste',xlabel='Data do alvo',ylabel='Cota (unidade da exportação ANA)'); ax.grid(alpha=.25); ax.legend(ncol=4); fig.tight_layout(); fig.savefig(DIR_FIGURAS/'sobreposicao_janelas.png',dpi=160,bbox_inches='tight'); plt.show()

# Erros por faixa estatística e resíduos com sinal para J7 com chuva até t.
v=pred_todas[7]; p=particoes[('t',7,True)]['treino']; p10,p90=np.quantile(p['y'],[.10,.90]); erro=v['pred']-v['real']
faixa=np.select([v['real']<=p10,v['real']>=p90],['Baixa (≤ P10 treino)','Alta (≥ P90 treino)'],'Central')
residuos=pd.DataFrame({'Data':v['datas'],'Real':v['real'],'Previsto':v['pred'],'Erro previsto−real':erro,'Faixa':faixa})
resumo_erros=residuos.groupby('Faixa',observed=True).apply(lambda g:pd.Series({'N':len(g),'MAE':np.mean(np.abs(g['Erro previsto−real'])),'RMSE':np.sqrt(np.mean(g['Erro previsto−real']**2)),'Erro médio':g['Erro previsto−real'].mean()}),include_groups=False).reset_index()
display(resumo_erros); resumo_erros.to_csv(DIR_RESULTADOS/'erros_por_faixa.csv',index=False); residuos.to_csv(DIR_RESULTADOS/'residuos_teste_j7.csv',index=False,date_format='%Y-%m-%d')
fig,ax=plt.subplots(figsize=(8,4)); ax.axhline(0,color='black',lw=.8); ax.scatter(residuos['Real'],residuos['Erro previsto−real'],s=12,alpha=.4); ax.axvline(p10,ls='--',color='gray'); ax.axvline(p90,ls='--',color='gray'); ax.set(title='Resíduos no teste — J7, chuva até t',xlabel='Cota observada',ylabel='Previsto − observado'); ax.grid(alpha=.2); fig.tight_layout(); fig.savefig(DIR_FIGURAS/'residuos_niveis.png',dpi=160,bbox_inches='tight'); plt.show()

# Acumulados de chuva em janelas completas, sem preencher dias ausentes.
chuva_acc={w:diaria['Chuva'].rolling(w,min_periods=w).sum() for w in (7,15,30,60)}
chuva_tabela=pd.DataFrame({'Data':v['datas'],'Cota':v['real']})
for w,s in chuva_acc.items(): chuva_tabela[f'Chuva_{w}d']=[s.get(d-pd.Timedelta(days=1),np.nan) for d in v['datas']]
chuva_tabela.to_csv(DIR_RESULTADOS/'chuva_antecedente_teste.csv',index=False,date_format='%Y-%m-%d')
fig,axes=plt.subplots(2,2,figsize=(12,8));
for ax,w in zip(axes.flat,(7,15,30,60)):
    ax.scatter(chuva_tabela[f'Chuva_{w}d'],chuva_tabela['Cota'],s=10,alpha=.35); ax.set(title=f'Chuva acumulada nos {w} dias anteriores',xlabel='Precipitação acumulada (unidade da exportação ANA)',ylabel='Cota (unidade da exportação ANA)'); ax.grid(alpha=.2)
fig.tight_layout(); fig.savefig(DIR_FIGURAS/'chuva_acumulada_niveis.png',dpi=160,bbox_inches='tight'); plt.show()

# Episódios contínuos acima/abaixo dos percentis do treino; somente intervalos com cota observada.
serie_cota=diaria['Cota']; mask=(serie_cota.notna() & (serie_cota>=p90)) | (serie_cota.notna() & (serie_cota<=p10))
episodios=[]; grupo=None; tipo=None; anterior=None
for data,valor in serie_cota.items():
    novo='Alta' if pd.notna(valor) and valor>=p90 else ('Baixa' if pd.notna(valor) and valor<=p10 else None)
    if novo is None or (grupo is not None and (novo!=tipo or data-anterior!=pd.Timedelta(days=1))):
        if grupo: episodios.append({'Faixa':tipo,'Início':grupo[0][0],'Fim':grupo[-1][0],'Duração dias':len(grupo),'Cota mínima':min(x[1] for x in grupo),'Cota máxima':max(x[1] for x in grupo)})
        grupo=None; tipo=None
    if novo is not None:
        if grupo is None: grupo=[]; tipo=novo
        grupo.append((data,float(valor))); anterior=data
if grupo: episodios.append({'Faixa':tipo,'Início':grupo[0][0],'Fim':grupo[-1][0],'Duração dias':len(grupo),'Cota mínima':min(x[1] for x in grupo),'Cota máxima':max(x[1] for x in grupo)})
episodios=pd.DataFrame(episodios); episodios.to_csv(DIR_RESULTADOS/'episodios_p10_p90.csv',index=False,date_format='%Y-%m-%d')
display(episodios[episodios['Início'].dt.year.isin([2020,2022,2023,2024])] if len(episodios) else episodios)

print(f'P10 treino={p10:.3f}; P90 treino={p90:.3f}. São limites estatísticos, não operacionais.')
print('O erro positivo indica superestimação; erro negativo indica subestimação.')

P10 treino=96.000; P90 treino=358.000. São limites estatísticos, não operacionais.
O erro positivo indica superestimação; erro negativo indica subestimação.


<ipython-input-6-8b034d80f935>:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set(title=f'Cota observada e previsão — janela {n} dias (teste exploratório)',xlabel='Data do alvo',ylabel='Cota (unidade da exportação ANA)'); ax.grid(alpha=.25); ax.legend(); fig.tight_layout(); fig.savefig(DIR_FIGURAS/f'comparacao_{n}_dias.png',dpi=160,bbox_inches='tight'); plt.show()
<ipython-input-6-8b034d80f935>:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.suptitle('Comparação das janelas — MLP com cotas e chuva'); fig.tight_layout(); fig.savefig(DIR_FIGURAS/'comparacao_janelas_metricas.png',dpi=160,bbox_inches='tight'); plt.show()
<ipython-input-6-8b034d80f935>:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set(title='Sobreposição das previsões nas mesmas datas de teste',xlabel='Data do alvo',ylabel='Cota (unidade da exportação ANA)'); ax.grid(alpha=.25); ax.legend(ncol=4); fig.tight_layout(); 

,Faixa,N,MAE,RMSE,Erro médio
0,Alta (≥ P90 treino),239.0,108.762499,189.277527,-46.747866
1,Baixa (≤ P10 treino),42.0,28.139492,30.709033,-26.885643
2,Central,1970.0,22.215384,35.422692,3.210031


,Faixa,Início,Fim,Duração dias,Cota mínima,Cota máxima
717,Alta,2020-06-30,2020-07-03,4,398.0,848.0
718,Alta,2020-07-06,2020-07-20,15,398.0,1909.0
719,Alta,2020-08-13,2020-08-18,6,364.0,606.0
729,Alta,2022-04-05,2022-04-06,2,381.0,392.0
730,Alta,2022-05-02,2022-05-07,6,453.0,1179.0
731,Alta,2022-05-30,2022-06-03,5,367.0,1139.0
732,Alta,2022-06-06,2022-06-10,5,395.0,1082.0
733,Alta,2022-06-17,2022-06-29,13,372.0,1008.0
734,Alta,2022-07-16,2022-07-18,3,365.0,415.0
735,Alta,2022-08-12,2022-08-12,1,401.0,401.0


## 6. Auditoria de unidade, evento e limita??es

A exporta??o ANA n?o declara a unidade de `CotaXX` nos metadados locais consultados. O levantamento do [SGB](https://rigeo.sgb.gov.br/bitstream/doc/25518/1/marcuzzo_silva_Mendes_zero%20ortometrico_artigo.pdf) documenta zero da r?gua de Encantado em 28,160 m, sem provar invari?ncia hist?rica. A marca nivelada de 23,14 m em 02/05/2024 foi reportada pelo [SGB](https://rigeo.sgb.gov.br/bitstream/doc/24939.10/6/2024-11-04_nota_tecnica_levantamento_cheia_2024_v.8.pdf); ela ? um levantamento do evento, n?o necessariamente a m?dia di?ria. Neste CSV, `MediaDiaria=1` n?o tem valor/status di?rio eleg?vel em 02/05, enquanto h? leitura com valor 2314 e status 1 em linha `MediaDiaria=0`.

Um [boletim SGB de 12/05/2024](https://www.sgb.gov.br/sace/boletins/Taquari/20240512_02-20240512%20-%20032619.pdf) apresenta 1.200 cm como cota de inunda??o para Encantado; esse valor ? contextual e datado. N?o foi aplicado ?s m?dias di?rias sem confirma??o de comparabilidade de refer?ncia e vig?ncia. P10/P90 s?o somente estat?sticos. Cota baixa n?o permite inferir racionamento. A chuva vem de uma esta??o e n?o representa toda a bacia; vaz?o pode ser derivada da cota.

In [7]:
# Manifesto de artefatos criados por esta execução.
arquivos_gerados=sorted(str(p.relative_to(RAIZ)) for p in DIR_RESULTADOS.rglob('*') if p.is_file())
arquivos_gerados.append(str((DIR_RESULTADOS/'manifesto_arquivos.txt').relative_to(RAIZ)))
(DIR_RESULTADOS/'manifesto_arquivos.txt').write_text('\n'.join(arquivos_gerados)+'\n',encoding='utf-8')
print('Figuras:'); print(*[p.name for p in sorted(DIR_FIGURAS.glob('*.png'))],sep='\n')
print('Reprodução: abrir este notebook no VS Code, selecionar o kernel do .venv e executar todas as células.')

Figuras:
chuva_acumulada_niveis.png
comparacao_15_dias.png
comparacao_30_dias.png
comparacao_7_dias.png
comparacao_janelas_metricas.png
residuos_niveis.png
sobreposicao_janelas.png
Reprodução: abrir este notebook no VS Code, selecionar o kernel do .venv e executar todas as células.
